# 03 — Pandas and NumPy exploratory analysis

The questions below use only columns present in the downloaded CSVs. Counts and descriptive results are computed when the notebook runs; the dataset is not treated as a complete census of all global movies.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.data_loader import load_datasets, PROCESSED_DATA_DIR
datasets = load_datasets(PROCESSED_DATA_DIR)
if not datasets:
    datasets = load_datasets(ROOT / 'data' / 'raw', require_all=True)
movies = datasets['movies']
movies.head()

## Business question: how many releases are recorded each year?
Release dates are parsed with errors set to missing. Rows with invalid or absent dates are excluded from this particular chart, not removed from the dataset.

In [ ]:
release_dates = pd.to_datetime(movies['release_date'], errors='coerce')
releases_by_year = release_dates.dt.year.value_counts().sort_index().rename_axis('year').rename('movies')
display(releases_by_year.to_frame().tail(20))
print(f"Movies with a usable release date: {release_dates.notna().sum():,} of {len(movies):,}.")

## Business question: what do ratings and runtime look like?
The table reports sample size alongside average, spread, and extremes. Extremely high/low observations should be investigated before interpreting them as errors.

In [ ]:
numeric_columns = [c for c in ['vote_average', 'vote_count', 'runtime', 'budget', 'revenue', 'popularity'] if c in movies]
descriptive = movies[numeric_columns].apply(pd.to_numeric, errors='coerce').describe().T
display(descriptive)
if {'budget', 'revenue'}.issubset(movies.columns):
    pairs = movies[['budget', 'revenue']].apply(pd.to_numeric, errors='coerce').dropna()
    pairs = pairs[(pairs['budget'] > 0) & (pairs['revenue'] > 0)]
    print(f'Pearson correlation from {len(pairs):,} positive budget/revenue pairs: {pairs.corr().iloc[0, 1]:.3f}')
    print('This is an association, not evidence that budget causes revenue.')

## Business question: which genres appear most often?
Genre names are joined through the bridge table; a movie assigned several genres contributes once to each of those genres.

In [ ]:
genre_data = datasets['movie_genres'].merge(datasets['genres'], on='genre_id', how='left')
genre_counts = genre_data.groupby('genre_name', dropna=False)['movie_id'].nunique().sort_values(ascending=False)
display(genre_counts.rename('unique_movies').to_frame())

## Data-quality context
Missingness can affect the available analysis. Below is the missing percentage by movie field; values are based on the table currently loaded.

In [ ]:
missing_summary = movies.isna().mean().mul(100).sort_values(ascending=False).rename('missing_percent')
display(missing_summary.to_frame())
print('Numeric values can include zero placeholders; budget/revenue profitability filters require positive values.')